# A06 - 简化流固耦合：通道压力载荷与弹性壁面

真实 FSI 会涉及移动边界、ALE 或 immersed boundary 等复杂处理。这个教程先抓住 PINN 中 FSI 的关键耦合结构：

- 流体给结构提供压力载荷 `p(x, y=1)`
- 结构位移 `w(x)` 满足梁方程 `D*w'''' = p_top(x)`
- 接口位移可进一步反馈给流体边界。本 notebook 先做单向耦合，再给出双向改造位置。

为了可运行，流场使用 Poiseuille 型解析压力 `p=1-x` 作为教学载荷，同时用 PINN 学习压力函数和梁位移。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(106)
device = "cuda" if torch.cuda.is_available() else "cpu"
D = 0.05
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width=64, depth=4):
        super().__init__()
        layers = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, out_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

pressure_net = MLP(2, 1).to(device)
beam_net = MLP(1, 1).to(device)

def exact_pressure(x, y):
    return 1 - x

In [ ]:
def pressure_loss(n=512):
    x = torch.rand(n, 1, device=device)
    y = torch.rand(n, 1, device=device)
    pred = pressure_net(torch.cat([x, y], dim=1))
    data = exact_pressure(x, y)
    return torch.mean((pred - data)**2)

def beam_residual(x):
    x.requires_grad_(True)
    w = beam_net(x)
    w1 = grad(w, x)
    w2 = grad(w1, x)
    w3 = grad(w2, x)
    w4 = grad(w3, x)
    y_top = torch.ones_like(x)
    p_top = pressure_net(torch.cat([x, y_top], dim=1))
    return D * w4 - p_top

def beam_bc_loss():
    x0 = torch.zeros(1, 1, device=device).requires_grad_(True)
    x1 = torch.ones(1, 1, device=device).requires_grad_(True)
    w0 = beam_net(x0); w1 = beam_net(x1)
    slope0 = grad(w0, x0); slope1 = grad(w1, x1)
    # clamped-clamped beam: w=0 and w'=0 at both ends
    return torch.mean(w0**2 + w1**2 + slope0**2 + slope1**2)

optimizer = torch.optim.Adam(list(pressure_net.parameters()) + list(beam_net.parameters()), lr=1e-3)
history = []
for epoch in range(10000):
    x = torch.rand(512, 1, device=device)
    loss_p = pressure_loss()
    loss_beam = torch.mean(beam_residual(x)**2)
    loss_bc = beam_bc_loss()
    loss = loss_p + loss_beam + 50 * loss_bc
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    history.append([loss.item(), loss_p.item(), loss_beam.item(), loss_bc.item()])
    if epoch % 2000 == 0:
        print(f"epoch={epoch:5d}, total={loss.item():.3e}, pressure={loss_p.item():.3e}, beam={loss_beam.item():.3e}")

In [ ]:
x = torch.linspace(0, 1, 300, device=device).reshape(-1, 1)
with torch.no_grad():
    p_top = pressure_net(torch.cat([x, torch.ones_like(x)], dim=1))
    w = beam_net(x)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.plot(x.cpu(), p_top.cpu(), label="learned p_top")
plt.plot(x.cpu(), (1-x).cpu(), "--", label="target")
plt.legend(); plt.title("fluid pressure load")
plt.subplot(1,3,2)
plt.plot(x.cpu(), w.cpu())
plt.title("beam displacement w(x)")
plt.subplot(1,3,3)
plt.semilogy(torch.tensor(history)[:,0], label="total")
plt.semilogy(torch.tensor(history)[:,2], label="beam")
plt.legend(); plt.title("loss")
plt.tight_layout(); plt.show()

## 如何改成双向 FSI

1. 将流体域上边界从 `y=1` 改成 `y=1+w(x)`。
2. 使用坐标映射 `eta = y/(1+w(x))` 把变形域映射到固定域。
3. 在流体边界加入运动学条件，例如壁面法向速度与结构速度一致。
4. 将压力和剪切应力共同作为结构载荷，而不只用压力。